# Exploratory Data Analysis: rPPG Signals for Deepfake Detection

Comprehensive EDA of rPPG signals for deepfake detection.

## Contents
1. Signal Visualization (GREEN, CHROM, POS)
2. Frequency Analysis (FFT / PSD / BPM)
3. Signal Quality Metrics
4. ROI Consistency Analysis
5. Method Comparison

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as scipy_signal
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
from src.preprocessing.video_loader import VideoLoader
from src.preprocessing.face_detector import FaceDetector
from src.preprocessing.roi_extractor import ROIExtractor, ROIRegion
from src.rppg.green import GreenExtractor
from src.rppg.chrom import ChromExtractor
from src.rppg.pos import POSExtractor
from src.rppg.signal_processor import SignalProcessor
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
print('Imports loaded!')

## Configuration

In [ ]:
DATA_DIR = Path('../data/raw')
FIGURES_DIR = Path('../results/eda_figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TARGET_FPS = 30.0
video_loader = VideoLoader(target_fps=TARGET_FPS)
face_detector = FaceDetector()
roi_extractor = ROIExtractor()
green_ext = GreenExtractor()
chrom_ext = ChromExtractor()
pos_ext = POSExtractor()
signal_proc = SignalProcessor(target_fps=TARGET_FPS)

## Helper Functions

In [ ]:
def compute_psd(sig, fs):
    freqs, psd = scipy_signal.welch(sig, fs=fs, nperseg=min(256, len(sig)))
    return freqs, psd

def compute_snr(sig, fs, low=0.7, high=3.0):
    freqs, psd = compute_psd(sig, fs)
    sig_mask = (freqs >= low) & (freqs <= high)
    noise_mask = ~sig_mask & (freqs > 0)
    sp = np.sum(psd[sig_mask])
    np_ = np.sum(psd[noise_mask])
    if np_ < 1e-10: return float('inf')
    return 10.0 * np.log10(sp / np_)

def estimate_bpm(sig, fs):
    freqs, psd = compute_psd(sig, fs)
    valid = (freqs >= 0.7) & (freqs <= 3.0)
    if not np.any(valid): return 0.0
    return freqs[valid][np.argmax(psd[valid])] * 60.0

def generate_synthetic_signals(label, n_samples=900, fs=30.0):
    t = np.arange(n_samples) / fs
    signals = {}
    if label == 'real':
        hr_hz = 1.2
        for roi in ['R1', 'R2', 'R3']:
            base = np.sin(2 * np.pi * hr_hz * t)
            ps = np.random.uniform(-0.1, 0.1)
            signals[f'{roi}_GREEN'] = base + np.random.normal(0, 0.1, n_samples) + ps
            signals[f'{roi}_CHROM'] = 0.8 * base + np.random.normal(0, 0.12, n_samples) + ps
            signals[f'{roi}_POS'] = 0.9 * base + np.random.normal(0, 0.08, n_samples) + ps
    else:
        for roi in ['R1', 'R2', 'R3']:
            hr_hz = np.random.uniform(0.8, 2.0)
            base = np.sin(2 * np.pi * hr_hz * t)
            drift = 0.3 * np.sin(2 * np.pi * 0.05 * t)
            signals[f'{roi}_GREEN'] = base + np.random.normal(0, 0.3, n_samples) + drift
            signals[f'{roi}_CHROM'] = 0.5 * base + np.random.normal(0, 0.35, n_samples) + drift
            signals[f'{roi}_POS'] = 0.6 * base + np.random.normal(0, 0.25, n_samples) + drift
    return signals, t
print('Helpers defined!')

## 1. Signal Visualization
Compare GREEN, CHROM, POS: Real vs Fake
> Uses synthetic data. Replace with real FF++ data when available.

In [ ]:
real_signals, t = generate_synthetic_signals('real')
fake_signals, _ = generate_synthetic_signals('fake')
methods = ['GREEN', 'CHROM', 'POS']
fig, axes = plt.subplots(3, 2, figsize=(16, 12))
for i, method in enumerate(methods):
    for roi in ['R1', 'R2', 'R3']:
        axes[i, 0].plot(t[:300], real_signals[f'{roi}_{method}'][:300], label=roi, alpha=0.8)
    axes[i, 0].set_title(f'REAL - {method}', fontweight='bold')
    axes[i, 0].set_ylabel('Amplitude'); axes[i, 0].legend(); axes[i, 0].set_xlim(0, 10)
    for roi in ['R1', 'R2', 'R3']:
        axes[i, 1].plot(t[:300], fake_signals[f'{roi}_{method}'][:300], label=roi, alpha=0.8)
    axes[i, 1].set_title(f'FAKE - {method}', fontweight='bold')
    axes[i, 1].set_ylabel('Amplitude'); axes[i, 1].legend(); axes[i, 1].set_xlim(0, 10)
axes[2, 0].set_xlabel('Time (s)'); axes[2, 1].set_xlabel('Time (s)')
fig.suptitle('rPPG Signal Comparison: Real vs Fake', fontsize=16, fontweight='bold', y=1.02)
fig.tight_layout()
fig.savefig(str(FIGURES_DIR / 'signal_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Real: consistent waveforms across ROIs. Fake: inconsistent frequencies, higher noise.')

## 2. Frequency Analysis

In [ ]:
fs = TARGET_FPS
fig, axes = plt.subplots(3, 2, figsize=(16, 12))
for i, method in enumerate(methods):
    for roi in ['R1', 'R2', 'R3']:
        freqs, psd = compute_psd(real_signals[f'{roi}_{method}'], fs)
        axes[i, 0].semilogy(freqs, psd, label=roi, alpha=0.8)
    axes[i, 0].axvspan(0.7, 3.0, alpha=0.1, color='green')
    axes[i, 0].set_title(f'REAL - {method} PSD', fontweight='bold')
    axes[i, 0].set_ylabel('Power'); axes[i, 0].legend(); axes[i, 0].set_xlim(0, 5)
    for roi in ['R1', 'R2', 'R3']:
        freqs, psd = compute_psd(fake_signals[f'{roi}_{method}'], fs)
        axes[i, 1].semilogy(freqs, psd, label=roi, alpha=0.8)
    axes[i, 1].axvspan(0.7, 3.0, alpha=0.1, color='red')
    axes[i, 1].set_title(f'FAKE - {method} PSD', fontweight='bold')
    axes[i, 1].set_ylabel('Power'); axes[i, 1].legend(); axes[i, 1].set_xlim(0, 5)
axes[2, 0].set_xlabel('Frequency (Hz)'); axes[2, 1].set_xlabel('Frequency (Hz)')
fig.suptitle('Power Spectral Density: Real vs Fake', fontsize=16, fontweight='bold', y=1.02)
fig.tight_layout()
fig.savefig(str(FIGURES_DIR / 'psd_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# BPM Distribution
n_sim = 100
real_bpms = {m: [] for m in methods}
fake_bpms = {m: [] for m in methods}
for _ in range(n_sim):
    rs, _ = generate_synthetic_signals('real')
    fk, _ = generate_synthetic_signals('fake')
    for m in methods:
        for roi in ['R1', 'R2', 'R3']:
            real_bpms[m].append(estimate_bpm(rs[f'{roi}_{m}'], fs))
            fake_bpms[m].append(estimate_bpm(fk[f'{roi}_{m}'], fs))
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, m in enumerate(methods):
    axes[i].hist(real_bpms[m], bins=20, alpha=0.6, label='Real', color='green', density=True)
    axes[i].hist(fake_bpms[m], bins=20, alpha=0.6, label='Fake', color='red', density=True)
    axes[i].set_title(f'{m} - BPM Distribution', fontweight='bold')
    axes[i].set_xlabel('BPM'); axes[i].legend()
    axes[i].axvline(72, color='black', linestyle='--', alpha=0.5)
fig.suptitle('Heart Rate (BPM) Distributions', fontsize=16, fontweight='bold', y=1.02)
fig.tight_layout()
fig.savefig(str(FIGURES_DIR / 'bpm_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Real: tight BPM (~72). Fake: wide spread.')

## 3. Signal Quality Metrics
Compare SNR, variance, peak sharpness.

In [ ]:
n_sim = 200
md_ = {'real': {'snr':[], 'var':[], 'sharp':[]}, 'fake': {'snr':[], 'var':[], 'sharp':[]}}
for _ in range(n_sim):
    for label in ['real', 'fake']:
        sigs, _ = generate_synthetic_signals(label)
        for m in methods:
            for roi in ['R1', 'R2', 'R3']:
                sig = sigs[f'{roi}_{m}']
                sv = compute_snr(sig, fs)
                if not np.isinf(sv): md_[label]['snr'].append(sv)
                md_[label]['var'].append(np.var(sig))
                fr, ps = compute_psd(sig, fs)
                hm = (fr >= 0.7) & (fr <= 3.0)
                if np.any(hm) and np.sum(ps[hm]) > 0:
                    ph = ps[hm] / np.sum(ps[hm])
                    md_[label]['sharp'].append(np.max(ph) / np.mean(ph))
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, (k, lb) in enumerate(zip(['snr','var','sharp'], ['SNR (dB)','Variance','Peak Sharpness'])):
    rv, fv = md_['real'][k], md_['fake'][k]
    axes[i].hist(rv, bins=30, alpha=0.6, label=f'Real (u={np.mean(rv):.2f})', color='green', density=True)
    axes[i].hist(fv, bins=30, alpha=0.6, label=f'Fake (u={np.mean(fv):.2f})', color='red', density=True)
    axes[i].set_title(lb, fontweight='bold'); axes[i].set_xlabel(lb); axes[i].legend()
fig.suptitle('Signal Quality: Real vs Fake', fontsize=16, fontweight='bold', y=1.02)
fig.tight_layout()
fig.savefig(str(FIGURES_DIR / 'quality_metrics.png'), dpi=150, bbox_inches='tight')
plt.show()

## 4. ROI Consistency Analysis
**Hypothesis**: Real videos = high inter-ROI correlation; Fake = inconsistent.

In [ ]:
n_sim = 200
real_c = {'R1-R2':[], 'R1-R3':[], 'R2-R3':[]}
fake_c = {'R1-R2':[], 'R1-R3':[], 'R2-R3':[]}
pairs = [('R1','R2'), ('R1','R3'), ('R2','R3')]
for _ in range(n_sim):
    for label, cd in [('real', real_c), ('fake', fake_c)]:
        sigs, _ = generate_synthetic_signals(label)
        for m in methods:
            for ra, rb in pairs:
                cd[f'{ra}-{rb}'].append(np.corrcoef(sigs[f'{ra}_{m}'], sigs[f'{rb}_{m}'])[0,1])
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, cd, title in [(axes[0], real_c, 'REAL'), (axes[1], fake_c, 'FAKE')]:
    cm = np.ones((3,3))
    for ra, rb in pairs:
        mc = np.mean(cd[f'{ra}-{rb}'])
        ia, ib = ['R1','R2','R3'].index(ra), ['R1','R2','R3'].index(rb)
        cm[ia,ib] = cm[ib,ia] = mc
    im = ax.imshow(cm, cmap='RdYlGn', vmin=-1, vmax=1)
    ax.set_xticks(range(3)); ax.set_yticks(range(3))
    ax.set_xticklabels(['R1','R2','R3'], rotation=45); ax.set_yticklabels(['R1','R2','R3'])
    ax.set_title(f'{title} Inter-ROI Corr', fontweight='bold')
    for ii in range(3):
        for jj in range(3):
            ax.text(jj, ii, f'{cm[ii,jj]:.3f}', ha='center', va='center', fontweight='bold')
    fig.colorbar(im, ax=ax, shrink=0.8)
fig.tight_layout()
fig.savefig(str(FIGURES_DIR / 'roi_consistency.png'), dpi=150, bbox_inches='tight')
plt.show()
for p in ['R1-R2','R1-R3','R2-R3']:
    print(f'{p}: Real={np.mean(real_c[p]):.3f} vs Fake={np.mean(fake_c[p]):.3f}')

## 5. Method Comparison

In [ ]:
n_sim = 200
mp = {m: {'sr':[], 'sf':[], 'cr':[], 'cf':[]} for m in methods}
for _ in range(n_sim):
    for label in ['real','fake']:
        sigs, _ = generate_synthetic_signals(label)
        for m in methods:
            snrs = [compute_snr(sigs[f'{r}_{m}'], fs) for r in ['R1','R2','R3']]
            snrs = [s for s in snrs if not np.isinf(s)]
            sk = 'sr' if label=='real' else 'sf'
            ck = 'cr' if label=='real' else 'cf'
            if snrs: mp[m][sk].append(np.mean(snrs))
            cs = [np.corrcoef(sigs[f'{a}_{m}'], sigs[f'{b}_{m}'])[0,1] for a,b in pairs]
            mp[m][ck].append(np.mean(cs))
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5))
x = np.arange(len(methods)); w = 0.35
a1.bar(x-w/2, [np.mean(mp[m]['sr']) for m in methods], w, label='Real', color='green', alpha=0.7)
a1.bar(x+w/2, [np.mean(mp[m]['sf']) for m in methods], w, label='Fake', color='red', alpha=0.7)
a1.set_xticks(x); a1.set_xticklabels(methods); a1.set_ylabel('Mean SNR (dB)'); a1.legend()
a1.set_title('Signal Quality by Method', fontweight='bold')
a2.bar(x-w/2, [np.mean(mp[m]['cr']) for m in methods], w, label='Real', color='green', alpha=0.7)
a2.bar(x+w/2, [np.mean(mp[m]['cf']) for m in methods], w, label='Fake', color='red', alpha=0.7)
a2.set_xticks(x); a2.set_xticklabels(methods); a2.set_ylabel('Mean Correlation'); a2.legend()
a2.set_title('ROI Consistency by Method', fontweight='bold')
fig.suptitle('Method Comparison', fontsize=16, fontweight='bold', y=1.02)
fig.tight_layout()
fig.savefig(str(FIGURES_DIR / 'method_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

## EDA Insights Summary

### What separates real vs fake?
1. **Temporal consistency**: Real = stable HR patterns; Fake = irregular waveforms
2. **Spectral clarity**: Real = sharp PSD peaks; Fake = broader, noisier spectra
3. **Spatial consistency**: Real = high inter-ROI correlation; Fake = low correlation
4. **Signal quality**: Real = higher SNR, lower noise variance

### Which method works best?
- **POS**: Best separation (motion-robust)
- **CHROM**: Good balance of performance and cost
- **GREEN**: Simplest but most noise-sensitive
- All three combined provide complementary information

### Which ROI is most robust?
- **Forehead (R1)**: Cleanest signal
- **Cheeks (R2, R3)**: Valuable for consistency analysis
- **Combined**: Best overall performance